In [1]:
import torch
import numpy as np
from tifffile import imwrite

from src.ssunet.datasets import BinomDataset, ValidationDataset
from src.ssunet.models import Bit2Bit
from src.ssunet.configs import MasterConfig, SSUnetData
from src.ssunet.constants import DEFAULT_CONFIG_PATH
from src.tools.gpuinference import gpu_patch_inference
from src.tools.tools import clear_vram

from pathlib import Path

# --- Configuration Loading ---
try:
    config = MasterConfig.from_config(DEFAULT_CONFIG_PATH)
    print(f"Successfully loaded configuration from: {DEFAULT_CONFIG_PATH}")
    print(f"Experiment directory: {config.target_path}")
    print(f"Config copied to: {config.target_path / Path(DEFAULT_CONFIG_PATH).name}")
except FileNotFoundError:
    print(f"ERROR: Default configuration file not found at '{DEFAULT_CONFIG_PATH}'. Please check the path.")
except Exception as e:
    print(f"ERROR loading configuration: {e}")

# --- Data Loading ---
print("Loading data...")
try:
    ssunet_training_data_source = config.path_config.load_data_only()
    print(f"  Training data source loaded. Shape: {ssunet_training_data_source.primary_data.shape}, Type: {type(ssunet_training_data_source.primary_data)}")

    ssunet_validation_data_source = config.path_config.load_reference_and_ground_truth()
    print(f"  Validation data source loaded.")
    print(f"    Reference (for input) shape: {ssunet_validation_data_source.primary_data.shape if ssunet_validation_data_source.primary_data is not None else 'N/A'}")
    print(f"    Ground Truth (for target) shape: {ssunet_validation_data_source.secondary_data.shape if ssunet_validation_data_source.secondary_data is not None else 'N/A'}")
except FileNotFoundError as e:
    print(f"ERROR: Data file not found: {e}. Check PATH configuration in your config file.")
except Exception as e:
    print(f"ERROR loading data: {e}")

# --- Dataset Configuration & Creation ---
print("Configuring and creating datasets...")
data_config = config.data_config
validation_config = data_config.validation_config

training_data = BinomDataset(ssunet_training_data_source, data_config, config.split_params)
print(f"  Training dataset created. Length: {len(training_data)}")

validation_data = ValidationDataset(ssunet_validation_data_source, validation_config)
print(f"  Validation dataset created. Length: {len(validation_data)}")

# --- Model Creation ---
print("Initializing model...")
model = Bit2Bit(config.model_config)

# --- Data Loaders ---
print("Creating data loaders...")
training_loader = config.loader_config.loader(training_data)
validation_loader = config.loader_config.loader(validation_data)

# --- Check Input Size (Optional Sanity Check) ---
try:
    sample_input_batch, sample_target_batch = next(iter(training_loader))
    print(f"  Sample input batch shape to model: {tuple(sample_target_batch.shape)}")
    print(f"  Sample target batch shape for loss: {tuple(sample_input_batch.shape)}")
except StopIteration:
    print("ERROR: Training loader is empty. Check dataset size and batch_size.")
except Exception as e:
    print(f"Error getting sample batch from training_loader: {e}")

# --- Model Training ---
print("Starting model training...")
trainer = config.trainer
try:
    trainer.fit(model, training_loader, validation_loader)
    print("Training completed.")
except Exception as e:
    print(f"An error occurred during training or subsequent steps: {e}")

print("Script finished.")

Successfully loaded configuration from: config.yml
Experiment directory: models\20250509_172725_e=120_p=32_d=[0]_test_run_v2_1200x32x256x256_skip=1_l=10_d=5_sf=32_ds=2at10_f=10.0_z=3_g=8_sd=0_b=tri_a=gelu
Config copied to: models\20250509_172725_e=120_p=32_d=[0]_test_run_v2_1200x32x256x256_skip=1_l=10_d=5_sf=32_ds=2at10_f=10.0_z=3_g=8_sd=0_b=tri_a=gelu\config.yml
Loading data...
  Training data source loaded. Shape: (3600, 512, 512), Type: <class 'numpy.ndarray'>
  Validation data source loaded.
    Reference (for input) shape: (255, 512, 512)
    Ground Truth (for target) shape: (255, 512, 512)
Configuring and creating datasets...
  Training dataset created. Length: 1200
  Validation dataset created. Length: 224
Initializing model...
Creating data loaders...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


  Sample input batch shape to model: (3, 1, 32, 256, 256)
  Sample target batch shape for loss: (3, 1, 32, 256, 256)
Starting model training...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type                             | Params | Mode 
-------------------------------------------------------------------------
0 | psnr_metric | PeakSignalNoiseRatio             | 0      | train
1 | ssim_metric | StructuralSimilarityIndexMeasure | 0      | train
2 | down_convs  | ModuleList                       | 9.6 M  | train
3 | up_convs    | ModuleList                       | 5.1 M  | train
4 | conv_final  | Sequential                       | 33     | train
-------------------------------------------------------------------------
14.7 M    Trainable params
0         Non-trainable params
14.7 M    Total params
58.718    Total estimated model params size (MB)
100       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

SystemExit: 0

c:\Users\HEQ\Projects\ssunet\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3678: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
trainer.save_checkpoint(config.train_config.default_root_dir / "model.ckpt")


In [7]:
from src.tools.tools import clear_vram
clear_vram()



In [5]:
from src.tools.gpuinference import gpu_patch_inference
import numpy as np
from tifffile import imwrite

output = gpu_patch_inference(
    model,
    data.primary_data.astype(np.float32),
    min_overlap=48,
    initial_patch_depth=64,
    device=config.device,
)

imwrite(config.train_config.default_root_dir / "input.tif", data.primary_data)
imwrite(config.train_config.default_root_dir / "inference.tif", output)

Inference #: 100%|██████████| 225/225 [02:53<00:00,  1.30it/s, vram_usage=188773.5 GB]


In [8]:
from src.tools.tools import group_metrics
from tifffile import imread


input_new = data.primary_data.astype(np.float32)[:512].copy()
output_new = output[:512].copy()
ground_truth = imread(config.path_config.ground_truth_file)[:512].astype(np.float32).copy()

group_metrics(input_new, output_new, ground_truth, config.train_config.default_root_dir)
